# Mario Kart Tracker v2: Data Scraping

## Tracks

The aim of this notebook is to scrape the tracks from Mario Kart 8 Deluxe and World and store them in a CSV file

Website(s) to scrape from: 
* Nintendo Website

(We could use the Mario Wiki like for the other notebooks, but let's mix it up a bit)

Required Libraries:
* `Selenium`
* `BeautifulSoup`
* `Pandas`

In [1]:
from bs4 import BeautifulSoup
from selenium import webdriver
import pandas as pd
from pathlib import Path

In [2]:
wd = Path().cwd()
wd = wd.parent

### Mario Kart 8 Deluxe

In [20]:
url = 'https://www.nintendo.com/sg/switch/aabp/course/index.html?srsltid=AfmBOop_vyLUnGFtkTSkk2JqRc9A8QDg9EYDWAlGaJa628hixHt5waPR'

def get_track_html(url):
    driver = webdriver.Firefox()
    driver.get(url)

    # The courses only appear when you scroll down a bit
    driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')

    track_html = driver.page_source

    driver.quit()

    return track_html


standard_html = get_track_html(url)
print(standard_html)

<html data-nsuid="70010000000153" lang="en-SG" data-device="desktop" data-browser="other" data-inframe="0" data-win_tablet="0" class="loader-type-default no-touchevents is-ncommon-ghdr-ua-win" style=""><head>
  <meta charset="UTF-8">
  <!-- Google Tag Manager -->
  <script type="text/javascript" async="" src="https://www.googletagmanager.com/gtag/destination?id=AW-10784433429&amp;cx=c&amp;gtm=4e65j1"></script><script src="https://connect.facebook.net/signals/config/4631496183582944?v=2.9.324&amp;r=stable&amp;domain=www.nintendo.com&amp;hme=af8aa31887db259becaf70277daef60bd8bc35c2df82c2acd4258de27ecac4b5&amp;ex_m=104%2C207%2C155%2C22%2C72%2C73%2C146%2C68%2C67%2C11%2C164%2C90%2C16%2C138%2C127%2C39%2C75%2C78%2C134%2C160%2C166%2C8%2C4%2C5%2C7%2C6%2C3%2C91%2C101%2C167%2C172%2C221%2C62%2C188%2C189%2C55%2C279%2C30%2C74%2C233%2C232%2C231%2C23%2C33%2C103%2C61%2C10%2C63%2C97%2C98%2C99%2C105%2C130%2C31%2C29%2C132%2C133%2C129%2C128%2C156%2C76%2C159%2C157%2C158%2C50%2C60%2C123%2C15%2C163%2C45%2C266

In [15]:
# We're looking for the names, which are helpfully contained within p tags with 
# the class "name"
track_soup = BeautifulSoup(track_html)

name_tags = track_soup.find_all('p', class_ = 'name')

names = [p.text for p in name_tags]

# There should be 48 here (we'll do DLC after)
print(f"{len(names)} course names found")

144 course names found


In [17]:
# Okay, there are some that shouldn't be there

# Remove duplicates
names = list(set(names))
print(f"{len(names)} course names found")

48 course names found


In [21]:
# Let's get the DLC names, too
dlc_url = 'https://mariokart8.nintendo.com/booster-course-pass/'

dlc_html = get_track_html(dlc_url)
print(dlc_html)

<html lang="en" locale="en-us" data-eshop-lang="en" data-eshop-country="us,ca" class="svg cssanimations flexboxlegacy fontface csstransforms supports csstransforms3d csstransitions inlinesvg alps-ua-firefox alps-os-windows"><head><style class="vjs-styles-defaults">
      .video-js {
        width: 300px;
        height: 150px;
      }

      .vjs-fluid {
        padding-top: 56.25%
      }
    </style>
  <meta charset="UTF-8">
  <meta http-equiv="X-UA-Compatible" content="IE=edge">
  <title>Mario Kart 8 Deluxe — Booster Course Pass for the Mario Kart 8 Deluxe game on the Nintendo Switch™ system — Official Site</title>

  <meta name="viewport" content="width=device-width,initial-scale=1,user-scalable=1">


    <meta property="og:title" content="Mario Kart 8 Deluxe — Booster Course Pass for the Mario Kart 8 Deluxe game on the Nintendo Switch™ system — Official Site">
    <meta property="og:description" content="Add a total of 48 more courses and 8 additional characters to the Mario Kart 

In [26]:
dlc_soup = BeautifulSoup(dlc_html)

dlc_name_spans = dlc_soup.find_all('span', class_ = 'plaque-image__plaque')
dlc_console_spans = dlc_soup.find_all('span', class_ = 'plaque-image__console')

dlc_names = []

for idx in range(0, len(dlc_name_spans)):
    name = dlc_name_spans[idx].text
    console = dlc_console_spans[idx].text

    if console:
        course_name = f"{name} ({console})"
    else:
        course_name = name

    dlc_names.append(course_name)

print(f"{len(dlc_names)} DLC cources found")


48 DLC cources found


In [27]:
# We've now found all the names for MK8!
# Just need to put them into their own dataframe

names.extend(dlc_names)

track_df = pd.DataFrame(names, columns=['name'])

track_df

,name
0,Excitebike Arena
1,Electrodrome
2,Water Park
3,Toad Harbor
4,SNES Donut Plains 3
...,...
91,Piranha Plant Cove
92,Madrid Drive (Tour)
93,Rosalina's Ice World (3DS)
94,Bowser Castle 3 (SNES)


### Mario Kart World

Now we can move onto MK World - hopefully it's as easy as 8

In [30]:
url = 'https://www.ign.com/wikis/mario-kart-world/Track_List'

driver = webdriver.Firefox()
driver.get(url)

world_html = driver.page_source

driver.quit()

print(world_html)

<html lang="en" data-build-id="KU_FytxpY_ELBBZM5olQ_" data-kraken-env="production" data-release="v0.97.49" data-theme="dark" color-scheme="light" style="color-scheme: dark;"><head prefix="og: http://ogp.me/ns# article: http://ogp.me/ns/article# fb: http://www.facebook.com/2008/fbml"><script async="" src="https://www.googletagmanager.com/gtm.js?id=GTM-K7G5DNRH"></script><script src="https://s0.2mdn.net/instream/video/client.js" async="" type="text/javascript"></script><script async="" data-jsonpid="" src="https://cdn-gl.imrworldwide.com/novms/js/2/nlsSDK600.bundle.min.js"></script><script async="" src="https://cdn-gl.imrworldwide.com/conf/config250.js#name=v60Bsdk__1779352521778&amp;ns=NOLBUNDLE"></script><script async="" src="https://sb.scorecardresearch.com/beacon.js"></script><script async="" 0="h" 1="t" 2="t" 3="p" 4="s" 5=":" 6="/" 7="/" 8="b" 9="-" 10="c" 11="o" 12="d" 13="e" 14="." 15="l" 16="i" 17="a" 18="d" 19="m" 20="." 21="c" 22="o" 23="m" 24="/" 25="a" 26="-" 27="0" 28="1" 2

In [40]:
# This page has the tracks in an unordered list so let's search for that!
world_soup = BeautifulSoup(world_html)

list_items = world_soup.find_all('li')

# We want the ones that do not contain any tags within them
list_items = [tag.text for tag in list_items if len(tag.find_all()) == 0]

# Remove duplicates
list_items = list(set(list_items))

list_items

['DK Pass',
 'Shy Guy Bazaar',
 'Koopa Troopa Beach',
 'Faraway Oasis',
 'Crown City',
 "Wario's Galleon",
 'Mario Bros. Circuit',
 'Great\xa0? Block Ruins',
 'Mario Circuit',
 'Peach Stadium (Version 1)',
 'Choco Mountain',
 'DK Spaceport',
 'Peach Stadium (Version 2)',
 'Sky-High Sundae',
 "Bowser's Castle",
 'Salty Salty Speedway',
 'Acorn Heights',
 'Cheep Cheep Falls',
 'Airship Fortress',
 'Boo Cinema',
 'Dry Bones Burnout',
 'Crown City (Version 2)',
 'Moo Moo Meadows',
 'Rainbow Road',
 'Starview Peak',
 'Dandelion Depths',
 'Desert Hills',
 'Peach Beach',
 'Peach Stadium',
 'Crown City (Version 1)',
 "Toad's Factory",
 'Whistlestop Summit',
 'Wario Galleon',
 'Dino Dino Jungle',
 'Wario Stadium']

We have all the tracks, but there are a couple of funny ones which we need to clean up

In [47]:
world_raw_df = pd.DataFrame(list_items, columns=['name'])

# Seems some had a couple of versions, so remove version numbers
world_raw_df['name'] = world_raw_df['name'].str.replace(pat=r' \(Version \d\)', repl='', regex=True)

world_raw_df = world_raw_df.drop_duplicates().sort_values(by='name')

print(f"{world_raw_df.shape[0]} tracks found")
world_raw_df

31 tracks found


,name
16,Acorn Heights
18,Airship Fortress
19,Boo Cinema
14,Bowser's Castle
17,Cheep Cheep Falls
10,Choco Mountain
4,Crown City
0,DK Pass
11,DK Spaceport
25,Dandelion Depths


In [51]:
# There's one double-up due to a misspelling. So we'll just delete it
world_clean_df = world_raw_df.copy()

world_clean_df = world_clean_df[world_clean_df['name'] != 'Wario Galleon']

world_clean_df = world_clean_df.reset_index(drop=True)

print(f"{world_clean_df.shape[0]} tracks found")
world_clean_df

30 tracks found


,name
0,Acorn Heights
1,Airship Fortress
2,Boo Cinema
3,Bowser's Castle
4,Cheep Cheep Falls
5,Choco Mountain
6,Crown City
7,DK Pass
8,DK Spaceport
9,Dandelion Depths


### Saving

Now we have all the tracks, so we can save them!

In [53]:
mk8_track_df = track_df.copy()
world_track_df = world_clean_df.copy()

mk8_track_df['game_version'] = 'Mario Kart 8 Deluxe'
world_track_df['game_version'] = 'Mario Kart World'

all_track_df = pd.concat([mk8_track_df, world_track_df], axis=0)

all_track_df

,name,game_version
0,Excitebike Arena,Mario Kart 8 Deluxe
1,Electrodrome,Mario Kart 8 Deluxe
2,Water Park,Mario Kart 8 Deluxe
3,Toad Harbor,Mario Kart 8 Deluxe
4,SNES Donut Plains 3,Mario Kart 8 Deluxe
...,...,...
25,Starview Peak,Mario Kart World
26,Toad's Factory,Mario Kart World
27,Wario Stadium,Mario Kart World
28,Wario's Galleon,Mario Kart World


In [54]:
all_track_df.to_csv(wd / 'data' / 'tracks.csv', index=False)